# ================================================
# Tour Place Recommender System
# Knowledge-Based Recommendation using Cosine Similarity
# Student: Khalid Mahmud Joy | ID: 2022-3-60-159
# ================================================

In [ ]:
# Cell 2 - Import Libraries
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cell 3 - Load Datasets
users_df       = pd.read_csv('Users.csv')
destinations_df = pd.read_csv('Destinations.csv')
ratings_df     = pd.read_csv('Ratings.csv')

print("Users shape:       ", users_df.shape)
print("Destinations shape:", destinations_df.shape)
print("Ratings shape:     ", ratings_df.shape)

Users shape:        (300, 6)
Destinations shape: (25, 9)
Ratings shape:      (2271, 3)


In [ ]:
# Cell 4 - Preview Data
print("=== Users ===")
display(users_df.head())

print("=== Destinations ===")
display(destinations_df.head())

print("=== Ratings ===")
display(ratings_df.head())

=== Users ===


,User-ID,Division,Age,Travel-Style,Preferred-Type,Budget-Level
0,1,Chattogram,19,Luxury,Nature,2
1,2,Rajshahi,26,Luxury,Forest,5
2,3,Chattogram,55,Nature,Beach,1
3,4,Chattogram,31,Relaxation,Beach,5
4,5,Rajshahi,63,Luxury,Lake,2


=== Destinations ===


,Dest-ID,Name,Division,Type,Tags,Budget-Level,Best-Season,Avg-Rating,Description
0,BD001,Cox's Bazar,Chattogram,Beach,beach sea sunset swimming resort seafood sandy...,3,Winter,9.1,World's longest natural sea beach stretching 1...
1,BD002,Sundarbans,Khulna,Forest,mangrove forest tiger wildlife boat safari jun...,4,Winter,9.3,"Largest mangrove forest, UNESCO World Heritage..."
2,BD003,Sajek Valley,Chattogram,Hill,hill valley cloud mist trekking tribal sunrise...,3,All,9.0,"Cloud-touching valley in Rangamati, called 'Qu..."
3,BD004,Srimangal,Sylhet,Nature,tea garden green nature bird watching forest l...,2,Winter,8.7,Tea capital of Bangladesh with lush green gard...
4,BD005,Saint Martin's Island,Chattogram,Beach,island coral reef snorkeling beach fishing boa...,4,Winter,9.2,Only coral island of Bangladesh with crystal c...


=== Ratings ===


,User-ID,Dest-ID,Rating
0,1,BD015,9
1,1,BD002,9
2,1,BD024,3
3,1,BD012,9
4,1,BD010,9


In [ ]:
# Cell 5 - Preprocessing
# Drop null values
users_df        = users_df.dropna()
destinations_df = destinations_df.dropna()
ratings_df      = ratings_df.dropna()

# Convert types
users_df['Age']          = users_df['Age'].astype(int)
users_df['Budget-Level'] = users_df['Budget-Level'].astype(int)

destinations_df['Budget-Level'] = destinations_df['Budget-Level'].astype(int)
destinations_df['Avg-Rating']   = destinations_df['Avg-Rating'].astype(float)

ratings_df['Rating'] = ratings_df['Rating'].astype(int)

# Convert string columns
str_cols_users = ['Division', 'Travel-Style', 'Preferred-Type']
for col in str_cols_users:
    users_df[col] = users_df[col].astype(str)

str_cols_dest = ['Name', 'Division', 'Type', 'Tags', 'Best-Season', 'Description']
for col in str_cols_dest:
    destinations_df[col] = destinations_df[col].astype(str)

print("Preprocessing complete!")
print(f"Users: {len(users_df)} | Destinations: {len(destinations_df)} | Ratings: {len(ratings_df)}")

Preprocessing complete!
Users: 300 | Destinations: 25 | Ratings: 2271


In [ ]:
# Cell 6 - Encode Dest-ID to numeric index for matrix
dest_id_to_index = {dest_id: idx for idx, dest_id in enumerate(destinations_df['Dest-ID'].unique())}
index_to_dest_id = {idx: dest_id for dest_id, idx in dest_id_to_index.items()}

user_id_to_index = {user_id: idx for idx, user_id in enumerate(users_df['User-ID'].unique())}
index_to_user_id = {idx: user_id for user_id, idx in user_id_to_index.items()}

ratings_df['user_idx'] = ratings_df['User-ID'].map(user_id_to_index)
ratings_df['dest_idx'] = ratings_df['Dest-ID'].map(dest_id_to_index)

print("Encoding complete!")
print(f"Total unique users: {len(user_id_to_index)}")
print(f"Total unique destinations: {len(dest_id_to_index)}")

Encoding complete!
Total unique users: 300
Total unique destinations: 25


In [ ]:
# Cell 7 - Build Sparse User-Destination Rating Matrix
num_users = len(user_id_to_index)
num_dests = len(dest_id_to_index)

sparse_matrix = coo_matrix(
    (ratings_df['Rating'].values,
     (ratings_df['user_idx'].values, ratings_df['dest_idx'].values)),
    shape=(num_users, num_dests)
)

# Convert to CSR for cosine similarity computation
csr_matrix = sparse_matrix.tocsr()

print(f"Sparse matrix shape: {csr_matrix.shape}")
print(f"Non-zero entries: {csr_matrix.nnz}")

Sparse matrix shape: (300, 25)
Non-zero entries: 2271


In [ ]:
# Cell 8 - Compute Cosine Similarity Between Users
user_similarity = cosine_similarity(csr_matrix)
similarity_df   = pd.DataFrame(user_similarity,
                               index=list(user_id_to_index.keys()),
                               columns=list(user_id_to_index.keys()))

print("User similarity matrix shape:", similarity_df.shape)
display(similarity_df.head())

User similarity matrix shape: (300, 300)


,1,2,3,4,5,6,7,8,9,10,...,291,292,293,294,295,296,297,298,299,300
1,1.000000,0.321328,0.714562,0.216333,0.350023,0.000000,0.268477,0.476137,0.571144,0.535288,...,0.455710,0.332038,0.206777,0.081076,0.315206,0.233224,0.603672,0.230308,0.350968,0.393146
2,0.321328,1.000000,0.235258,0.100604,0.081869,0.056521,0.117917,0.548327,0.189642,0.384712,...,0.546432,0.250832,0.326943,0.128193,0.080907,0.307934,0.000000,0.000000,0.268514,0.102626
3,0.714562,0.235258,1.000000,0.150727,0.247675,0.000000,0.152883,0.467545,0.813088,0.605940,...,0.289407,0.285717,0.235496,0.277011,0.233107,0.438131,0.276587,0.149883,0.275072,0.312580
4,0.216333,0.100604,0.150727,1.000000,0.591541,0.319639,0.294199,0.278049,0.210290,0.460726,...,0.211183,0.468637,0.000000,0.230994,0.277759,0.322491,0.229931,0.072683,0.136965,0.497465
5,0.350023,0.081869,0.247675,0.591541,1.000000,0.407004,0.095765,0.144428,0.369150,0.485324,...,0.278450,0.333992,0.172099,0.000000,0.283857,0.000000,0.213843,0.130971,0.460374,0.302429


In [ ]:
# Cell 9 - Knowledge-Based Scoring Function
# Match destination to user profile:
# Type match = 40%, Budget match = 30%, Tag/Travel-Style match = 30%

def knowledge_score(user_row, dest_row):
    score = 0.0

    # Type match (40%)
    if user_row['Preferred-Type'].lower() == dest_row['Type'].lower():
        score += 0.4

    # Budget match (30%) — within ±1 level is acceptable
    if abs(user_row['Budget-Level'] - dest_row['Budget-Level']) <= 1:
        score += 0.3

    # Travel style match with tags (30%)
    travel_style = user_row['Travel-Style'].lower()
    tags         = dest_row['Tags'].lower()
    if travel_style in tags:
        score += 0.3

    return score

In [ ]:
# Cell 10 - Collaborative Filtering: Find Similar Users
def get_similar_users(target_user_id, top_n=5):
    if target_user_id not in similarity_df.index:
        print(f"User {target_user_id} not found!")
        return []

    sim_scores = similarity_df[target_user_id].drop(index=target_user_id)
    top_users  = sim_scores.sort_values(ascending=False).head(top_n).index.tolist()
    return top_users

In [ ]:
# Cell 11 - Recommendation Function (Knowledge-Based + Collaborative)
def recommend_destinations(target_user_id, top_n_similar_users=5, top_n_recommendations=10):

    if target_user_id not in user_id_to_index:
        print(f"User ID {target_user_id} not found in dataset.")
        return

    print(f"\n{'='*60}")
    print(f"  Recommendations for User: {target_user_id}")
    print(f"{'='*60}")

    # Get user profile
    user_row = users_df[users_df['User-ID'] == target_user_id].iloc[0]
    print(f"\n  User Profile:")
    print(f"  Division       : {user_row['Division']}")
    print(f"  Age            : {user_row['Age']}")
    print(f"  Travel Style   : {user_row['Travel-Style']}")
    print(f"  Preferred Type : {user_row['Preferred-Type']}")
    print(f"  Budget Level   : {user_row['Budget-Level']}")

    # Destinations already rated by target user
    already_rated = ratings_df[ratings_df['User-ID'] == target_user_id]['Dest-ID'].tolist()

    # Get top similar users
    similar_users = get_similar_users(target_user_id, top_n=top_n_similar_users)
    print(f"\n  Top {top_n_similar_users} Similar Users: {similar_users}")

    # Collect candidate destinations from similar users (high ratings only ≥ 7)
    candidate_dest_ids = set()
    for sim_user in similar_users:
        sim_ratings = ratings_df[
            (ratings_df['User-ID'] == sim_user) &
            (ratings_df['Rating'] >= 7) &
            (~ratings_df['Dest-ID'].isin(already_rated))
        ]['Dest-ID'].tolist()
        candidate_dest_ids.update(sim_ratings)

    if not candidate_dest_ids:
        print("\n  Not enough collaborative data. Falling back to knowledge-based only.")
        candidate_dest_ids = set(destinations_df['Dest-ID'].tolist()) - set(already_rated)

    # Score each candidate destination
    scored_destinations = []
    for dest_id in candidate_dest_ids:
        dest_rows = destinations_df[destinations_df['Dest-ID'] == dest_id]
        if dest_rows.empty:
            continue
        dest_row = dest_rows.iloc[0]

        k_score   = knowledge_score(user_row, dest_row)
        avg_r     = dest_row['Avg-Rating'] / 10.0        # normalize to 0-1
        # Final score: 50% knowledge match + 30% avg rating + 20% collaborative signal
        final_score = (0.5 * k_score) + (0.3 * avg_r) + 0.2

        scored_destinations.append({
            'Dest-ID'     : dest_id,
            'Name'        : dest_row['Name'],
            'Type'        : dest_row['Type'],
            'Division'    : dest_row['Division'],
            'Budget-Level': dest_row['Budget-Level'],
            'Best-Season' : dest_row['Best-Season'],
            'Avg-Rating'  : dest_row['Avg-Rating'],
            'Score'       : round(final_score, 4)
        })

    # Sort by final score
    results_df = pd.DataFrame(scored_destinations).sort_values('Score', ascending=False).head(top_n_recommendations)
    results_df = results_df.reset_index(drop=True)
    results_df.index += 1

    print(f"\n  Top {top_n_recommendations} Recommended Destinations:\n")
    display(results_df[['Dest-ID', 'Name', 'Type', 'Division', 'Budget-Level', 'Best-Season', 'Avg-Rating', 'Score']])
    return results_df

In [ ]:
# Cell 12 - Run Recommendation for a Target User
target_user = int(input("Enter User-ID to get recommendations: "))
results = recommend_destinations(target_user_id=target_user,
                                  top_n_similar_users=5,
                                  top_n_recommendations=10)

Enter User-ID to get recommendations: 4

  Recommendations for User: 4

  User Profile:
  Division       : Chattogram
  Age            : 31
  Travel Style   : Relaxation
  Preferred Type : Beach
  Budget Level   : 5

  Top 5 Similar Users: [94, 114, 207, 128, 108]

  Top 10 Recommended Destinations:



,Dest-ID,Name,Type,Division,Budget-Level,Best-Season,Avg-Rating,Score
1,BD001,Cox's Bazar,Beach,Chattogram,3,Winter,9.1,0.673
2,BD023,Patenga Beach,Beach,Chattogram,1,All,7.6,0.628
3,BD012,Rangamati,Lake,Chattogram,3,Winter,8.9,0.467
4,BD004,Srimangal,Nature,Sylhet,2,Winter,8.7,0.461
5,BD021,Kaptai Lake,Lake,Chattogram,2,Winter,8.7,0.461
6,BD016,Sylhet City,City,Sylhet,2,All,8.2,0.446
7,BD013,Lalbagh Fort,Historical,Dhaka,1,All,8.1,0.443


In [ ]:
# Cell 13 - Evaluate: Top Similar Users Detail
def show_similar_users_detail(target_user_id, top_n=5):
    similar_users = get_similar_users(target_user_id, top_n=top_n)
    print(f"\nTop {top_n} Users Most Similar to User {target_user_id}:\n")
    for u in similar_users:
        sim_score = similarity_df.loc[target_user_id, u]
        urow = users_df[users_df['User-ID'] == u]
        if not urow.empty:
            urow = urow.iloc[0]
            print(f"  User {u} | Similarity: {sim_score:.4f} | "
                  f"Style: {urow['Travel-Style']} | "
                  f"Preferred: {urow['Preferred-Type']} | "
                  f"Budget: {urow['Budget-Level']}")

show_similar_users_detail(target_user, top_n=5)


Top 5 Users Most Similar to User 4:

  User 94 | Similarity: 0.8607 | Style: Luxury | Preferred: Hill | Budget: 5
  User 114 | Similarity: 0.6614 | Style: Adventure | Preferred: Historical | Budget: 4
  User 207 | Similarity: 0.6547 | Style: Luxury | Preferred: Forest | Budget: 4
  User 128 | Similarity: 0.6413 | Style: Luxury | Preferred: Forest | Budget: 5
  User 108 | Similarity: 0.6297 | Style: Luxury | Preferred: Nature | Budget: 5
